# OSM Walking Network Dataset for ArcGIS Pro
## Building a topologically correct pedestrian Network Dataset from OpenStreetMap data

This notebook produces a walking travel mode Network Dataset for ArcGIS Pro Network Analyst
using OpenStreetMap data sourced from Geofabrik. The output supports pedestrian routing
and Service Area (walk catchment) analysis for any Australian city.

The output Network Dataset includes:
- Topologically correct walking edges derived from OpenStreetMap
- WalkTime cost in minutes (5 km/h base, surface-adjusted on dedicated walking infrastructure)
- Correct bridge and underpass handling via osmnx node topology
- 97%+ network connectivity verified before GDB import

---

### Prerequisites

| Requirement | Notes |
|---|---|
| ArcGIS Pro 3.x | With **Network Analyst** extension licensed |
| ArcGIS Pro cloned conda environment | Clone the default `arcgispro-py3` env before installing packages |
| `osmnx` | `pip install osmnx` in cloned env |
| `osmconvert` | Download `osmconvert64.exe` from https://wiki.openstreetmap.org/wiki/Osmconvert#Windows |
| Australia PBF file | Download from https://download.geofabrik.de/australia-oceania/australia.html |
| `osm_walk_nd_template.xml` | Included with this notebook — keep in the same folder |
| RAM | 16 GB minimum; 32 GB+ recommended for large city bounding boxes |
| Disk space | ~5 GB working space for intermediate files |

Install packages via the **Python Command Prompt** (installed with ArcGIS Pro):
```
pip install osmnx
```

### Estimated Runtime

| Step | Approximate Time |
|---|---|
| osmconvert clip to city bbox | 3–5 minutes |
| osmnx graph build | 10–20 minutes (varies with city size) |
| Graph filtering and component extraction | 2–5 minutes |
| Attribute calculation and GDB import | 5–10 minutes |
| Network Dataset create and build | 2–5 minutes |
| **Total** | **~25–45 minutes** |

---

> **⚠ WARNING:** This notebook requires ArcGIS Pro with the Network Analyst extension.
> Cells using `arcpy.na` will raise an `ExecuteError` without this extension licensed and active.


In [1]:
# =============================================================================
# CONFIGURATION
# Edit the values in this section. All other cells run unchanged.
# =============================================================================

# City name -- used in output file and geodatabase names (no spaces)
CITY_NAME = "Sydney"

# Bounding box in WGS84 decimal degrees: (west, south, east, north)
# Tip: https://boundingbox.klokantech.com -- draw your area, export as CSV
# BBOX = (147.0, -43.2, 147.6, -42.65)     # Greater Hobart
# BBOX = (151.73, -28.74, 153.73, -25.74)  # South East Queensland
BBOX = (150.5, -34.2, 151.5, -33.3)        # Greater Sydney

# Output folder -- the GDB and all intermediate files are created here.
# The folder must already exist; the GDB is created automatically.
WORK_DIR = r"C:\GIS\WalkNetwork"

# ---- Input files -------------------------------------------------------

# Australia PBF -- download once from Geofabrik and reuse for any city.
# Rename the downloaded file to australia-latest.osm.pbf or update this path.
# https://download.geofabrik.de/australia-oceania/australia.html
AUSTRALIA_PBF = r"C:\GIS\WalkNetwork\australia-latest.osm.pbf"

# osmconvert executable -- standalone Windows binary, no install required.
# Download osmconvert64.exe and place it anywhere accessible.
# https://wiki.openstreetmap.org/wiki/Osmconvert#Windows
OSMCONVERT_EXE = r"C:\GIS\WalkNetwork\osmconvert64.exe"

# ---------------------------------------------------------------------------
# DERIVED PATHS -- do not edit below this line
# ---------------------------------------------------------------------------
import os

GDB           = os.path.join(WORK_DIR, f"{CITY_NAME}_Walk_Network.gdb")
FDS_NAME      = "OSM_Transportation"
FC_EDGES_NAME = "OSM_Walk_Segments"   # must match the template XML source name
ND_NAME       = "OSM_Walk_ND"

CLIPPED_OSM  = os.path.join(WORK_DIR, f"{CITY_NAME.lower()}_walk.osm")
TEMP_GPKG    = os.path.join(WORK_DIR, f"{CITY_NAME.lower()}_walk.gpkg")

# Network Dataset template -- must be in the same folder as this notebook
TEMPLATE_XML = os.path.join(os.getcwd(), "osm_walk_nd_template.xml")

# ---------------------------------------------------------------------------
# PROJECTION -- auto-derived from bounding box centre longitude
# ---------------------------------------------------------------------------
# GDA2020 MGA is the current Australian standard (all states adopted by 2020).
# Zones cover 6 degree longitude bands; EPSG = 7800 + zone number.
#
#   City          Lon        Zone  EPSG
#   Perth         115.9 E    50    7850
#   Darwin        130.8 E    52    7852
#   Adelaide      138.6 E    54    7854
#   Melbourne     145.0 E    55    7855
#   Hobart        147.3 E    55    7855
#   Canberra      149.1 E    55    7855
#   Sydney        151.2 E    56    7856
#   Brisbane/SEQ  153.0 E    56    7856
#
# The auto-calculation below is correct for all Australian cities.
# Override EPSG only if you have a specific reason:
#   GDA94 MGA (legacy):  EPSG = 28300 + zone  (e.g. 28356 for Zone 56)
#   Victoria state-wide: EPSG = 7899  (VicGrid2020, Lambert Conformal Conic)
#   Web Mercator:        EPSG = 3857  (not recommended -- distorts distances
#     by 20-35% at Australian latitudes, producing incorrect WalkTime values)
_center_lon = (BBOX[0] + BBOX[2]) / 2
_mga_zone   = int((_center_lon + 180) / 6) + 1
EPSG        = 7800 + _mga_zone  # GDA2020 MGA -- override here if needed

# ---- Walk model parameters ---------------------------------------------

# Base walk speed in km/h -- used for road centrelines
# Dedicated walking infrastructure uses surface-adjusted speeds (see Step 6)
DEFAULT_WALK_KMH = 5.0

# Maximum segment length to retain (metres)
# Removes extremely long rural tracks that inflate solver time
MAX_SEGMENT_M = 5000

print(f"City:       {CITY_NAME}")
print(f"Bbox:       {BBOX}")
print(f"Projection: GDA2020 MGA Zone {_mga_zone}  (EPSG:{EPSG})")
print(f"GDB:        {GDB}")
print(f"Template:   {TEMPLATE_XML}")
print(f"Output:     {FC_EDGES_NAME}  in  {FDS_NAME}")


City:       Sydney
Bbox:       (150.5, -34.2, 151.5, -33.3)
Projection: GDA2020 MGA Zone 56  (EPSG:7856)
GDB:        C:\Work\Brisbane_GTFS\SEQ_Network\Sydney_Walk_Network.gdb
Template:   C:\Users\Admin\Documents\ArcGIS\Projects\BrisbaneGTFS\osm_walk_nd_template.xml
Output:     OSM_Walk_Segments  in  OSM_Transportation


## Step 1: Verify Environment

Run this cell first to confirm all required packages are installed and accessible in your
ArcGIS Pro cloned conda environment. If any import fails, install the missing package
via the Python Command Prompt before continuing.

In [2]:
import sys
import importlib

print(f"Python: {sys.version}")
print()

checks = [
    ("osmnx",    "osmnx",    "__version__"),
    ("geopandas","geopandas","__version__"),
    ("networkx", "networkx", "__version__"),
    ("pandas",   "pandas",   "__version__"),
    ("numpy",    "numpy",    "__version__"),
    ("arcpy",    "arcpy",    None),
]

all_ok = True
for name, module, attr in checks:
    try:
        m = importlib.import_module(module)
        if module == "arcpy":
            ver = m.GetInstallInfo()["Version"]
        else:
            ver = getattr(m, attr)
        print(f"  OK  {name:<12} {ver}")
    except ImportError:
        print(f"  MISSING  {name}")
        all_ok = False

print()
if all_ok:
    print("All packages ready. Proceed to Step 2.")
else:
    print("Install missing packages in your cloned ArcGIS Pro conda environment:")
    print("  pip install osmnx")

Python: 3.13.7 | packaged by Anaconda, Inc. | (main, Sep 11 2025, 16:13:29) [MSC v.1938 64 bit (AMD64)]

  OK  osmnx        2.1.0
  OK  geopandas    1.1.0
  OK  networkx     3.4.2
  OK  pandas       2.3.0
  OK  numpy        2.2.0
  OK  arcpy        3.6.2

All packages ready. Proceed to Step 2.


## Step 2: Download the Australia PBF File

Download the full Australia OSM extract from Geofabrik:

1. Go to: https://download.geofabrik.de/australia-oceania/australia.html
2. Click **australia-latest.osm.pbf**
3. Save to the path set in `AUSTRALIA_PBF` in the CONFIG cell

The file is approximately 900 MB. Download it once and reuse it — Step 3 clips it
to your city area so you never need to process the full file again.

The cell below checks the file exists and is a plausible size. Run it to confirm
before continuing.

In [3]:
import os

if not os.path.exists(AUSTRALIA_PBF):
    raise FileNotFoundError(
        f"PBF file not found: {AUSTRALIA_PBF}\n"
        "Download from: https://download.geofabrik.de/australia-oceania/australia.html"
    )

size_mb = os.path.getsize(AUSTRALIA_PBF) / 1024 / 1024
print(f"Found: {AUSTRALIA_PBF}")
print(f"Size:  {size_mb:.0f} MB")

if size_mb < 200:
    print("WARNING: File is smaller than expected. It may be incomplete or a sub-region file.")
else:
    print("File size looks correct. Ready to proceed.")

Found: C:\Work\Brisbane_GTFS\Incoming\australia-260330.osm.pbf
Size:  886 MB
File size looks correct. Ready to proceed.


## Step 3: Clip to City Bounding Box (osmconvert)

**Why clip with osmconvert before loading into osmnx?**

The Australia PBF is ~900 MB and contains the entire continent. Loading it directly
into osmnx would require ~10 GB RAM and take over an hour. osmconvert clips it to
your city bounding box in 3–5 minutes, reducing the file to ~100–300 MB before
osmnx processes it.

The `--complete-ways` flag retains ways that cross the bbox boundary rather than
cutting them mid-segment. Without this, roads at the edge of the study area would
have broken endpoints, creating dead-end topology at the boundary.

**Download osmconvert:**
https://wiki.openstreetmap.org/wiki/Osmconvert#Windows

Save the `.exe` to the path set in `OSMCONVERT_EXE` in the CONFIG cell.

> **Note:** osmconvert writes temporary files to its working directory.
> This cell sets `cwd=WORK_DIR` explicitly — without this, osmconvert will
> fail with a temp file permission error if the current directory is read-only.

In [4]:
import subprocess
import os

if not os.path.exists(OSMCONVERT_EXE):
    raise FileNotFoundError(
        f"osmconvert not found: {OSMCONVERT_EXE}\n"
        "Download from: https://wiki.openstreetmap.org/wiki/Osmconvert#Windows"
    )

xmin, ymin, xmax, ymax = BBOX

cmd = [
    OSMCONVERT_EXE,
    AUSTRALIA_PBF,
    f"-b={xmin},{ymin},{xmax},{ymax}",
    "--complete-ways",           # include ways that cross the bbox boundary
    "--complete-multipolygons",
    "-o=" + CLIPPED_OSM
]

print(f"Clipping to bbox: {BBOX}")
print(f"Output:           {CLIPPED_OSM}")
print("Running osmconvert — expect 3–5 minutes...\n")

result = subprocess.run(
    cmd,
    capture_output = True,
    text            = True,
    cwd             = WORK_DIR   # osmconvert writes temp files to cwd
)

if result.returncode != 0:
    raise RuntimeError(f"osmconvert failed:\n{result.stderr}")

size_mb = os.path.getsize(CLIPPED_OSM) / 1024 / 1024
print(f"Done.")
print(f"Output: {CLIPPED_OSM}")
print(f"Size:   {size_mb:.0f} MB  (Australia PBF was {os.path.getsize(AUSTRALIA_PBF)/1024/1024:.0f} MB)")
print("\nProceed to Step 4.")

Clipping to bbox: (150.5, -34.2, 151.5, -33.3)
Output:           C:\Work\Brisbane_GTFS\SEQ_Network\sydney_walk.osm
Running osmconvert — expect 3–5 minutes...

Done.
Output: C:\Work\Brisbane_GTFS\SEQ_Network\sydney_walk.osm
Size:   1702 MB  (Australia PBF was 886 MB)

Proceed to Step 4.


## Step 4: Build Topological Graph (osmnx)

**Why osmnx `graph_from_xml` instead of reading the OSM file directly with GDAL/pyogrio?**

GDAL reads OSM files as raw linestrings — it does **not** resolve topology.
Streets that cross in the OSM data do not share nodes in the output. Footways drawn
alongside roads appear as isolated parallel lines with no connection between them.
A pedestrian on a footway cannot reach the adjacent road, and Service Area solves
either fail or produce tiny isolated catchments.

osmnx reads the OSM file and builds a proper NetworkX directed graph where:
- Every intersection becomes a **shared node** — edges are split there
- Bridges and underpasses have **separate, disconnected nodes** — no false junctions
- The result is topologically correct before any ArcGIS processing begins

The `useful_tags_way` setting tells osmnx to retain the `surface`, `foot`, `layer`,
`bridge`, and `tunnel` tags used in later steps for WalkTime adjustment and
elevation level calculation.

**This step takes 10–20 minutes for a large city bounding box.** The `log_console = True` setting
shows progress during the parse.

In [5]:
import osmnx as ox

# Retain additional OSM tags beyond the osmnx defaults
# surface, foot, layer, bridge, tunnel are used in Steps 5 and 6
ox.settings.useful_tags_way = [
    "bridge", "tunnel", "oneway", "lanes", "ref", "name",
    "highway", "maxspeed", "service", "access", "area",
    "landuse", "width", "est_width", "junction", "surface",
    "foot", "sidewalk", "crossing", "layer"
]
ox.settings.log_console = True

print(f"Building topological walk graph from: {CLIPPED_OSM}")
print(f"Expect 10–20 minutes for {CITY_NAME}-scale data...\n")

G = ox.graph_from_xml(
    CLIPPED_OSM,
    retain_all = False,  # drop isolated nodes not connected to any edge
    simplify   = True    # merge redundant intermediate nodes
)

nodes, edges = ox.graph_to_gdfs(G)

print(f"\nGraph built.")
print(f"  Nodes: {len(nodes):,}")
print(f"  Edges: {len(edges):,}")
print(f"  CRS:   {edges.crs}")
print(f"\nEdge columns: {list(edges.columns)}")

Building topological walk graph from: C:\Work\Brisbane_GTFS\SEQ_Network\sydney_walk.osm
Expect 10–20 minutes for Sydney-scale data...


Graph built.
  Nodes: 820,239
  Edges: 2,713,730
  CRS:   epsg:4326

Edge columns: ['osmid', 'highway', 'surface', 'oneway', 'reversed', 'length', 'geometry', 'name', 'maxspeed', 'service', 'layer', 'lanes', 'bridge', 'junction', 'crossing', 'ref', 'foot', 'sidewalk', 'access', 'landuse', 'area', 'width', 'tunnel', 'est_width']


## Step 5: Filter to Walking Whitelist and Extract Main Component

**Why a strict whitelist instead of an exclude list?**

An exclude list (removing motorways, railways, etc.) leaves hundreds of edge cases:
railway platform outlines, building footprint traces, canal boundary lines, and
construction zones all carry `highway` tags in OSM and slip through. A strict
whitelist -- only explicitly walkable types -- eliminates this entire class of problem.

**A note on OSM's `highway` key**

In OpenStreetMap, `highway` is the tag used for **all** road and path infrastructure --
not just motor roads. Steps, footways, cycleways, tracks, corridors, and bridleways
are all stored under the `highway` key alongside road types like `residential` and
`primary`. This is OSM convention, not an error.

The highway distribution printed at the end of this step reflects that: you will see
pedestrian infrastructure types (`steps`, `footway`, `cycleway`, `corridor`, `track`)
listed alongside road types. All of these are valid walkable `highway` values that
belong in the network.

**Why extract the largest connected component?**

After filtering to walkable edges, some segments become disconnected from the main
network -- isolated sub-graphs in parks, private estates, or areas with OSM data gaps.

For transit catchment analysis we only want the **main connected component**. If a
transit stop snaps to an isolated island of disconnected edges, the Service Area
solver returns a tiny polygon or fails entirely -- the stranded pedestrian problem.
Discarding isolated components before building the Network Dataset prevents this.

The **coverage percentage** is the key quality metric: 97%+ means the network is
well connected and the source data is solid. Below 80% indicates a topology problem
worth investigating before proceeding.

In [6]:
import networkx as nx
import geopandas as gpd
import pandas as pd

# Strict walkable highway whitelist -- anything not listed is excluded.
# In OSM, the 'highway' key covers ALL road/path types, not just motor roads.
# Each value here is a valid OSM highway tag for pedestrian-accessible infrastructure.
WALK_WHITELIST = {
    # Dedicated pedestrian / shared-path infrastructure
    "footway", "path", "pedestrian", "steps",
    "corridor", "elevator", "bridleway",
    # Shared-use roads (pedestrians legally walk here)
    "living_street", "residential", "service", "unclassified",
    "tertiary", "tertiary_link",
    "secondary", "secondary_link",
    "primary", "primary_link",
    # Shared cycling/pedestrian routes and unpaved tracks
    "cycleway", "track",
}

# Types where surface speed adjustment applies (Step 6).
# These are dedicated off-road / pedestrian-only infrastructure types.
WALK_INFRA = {"footway", "path", "pedestrian", "steps",
              "cycleway", "track", "bridleway"}

# ---------------------------------------------------------------------------
# PARSE OSM HIGHWAY TAG
# osmnx stores some highway values as Python lists when a way carries multiple
# highway tags (e.g. ["residential", "living_street"]). Take the first valid
# string value so all downstream operations work with a plain string.
# ---------------------------------------------------------------------------
def normalise_val(val):
    if isinstance(val, list):
        for v in val:
            if isinstance(v, str) and v.strip():
                return v.strip()
        return ""
    return str(val).strip() if val is not None else ""

edges_work = edges.copy().reset_index()
edges_work["highway_clean"] = edges_work["highway"].apply(normalise_val)

# ---------------------------------------------------------------------------
# WHITELIST FILTER
# ---------------------------------------------------------------------------
before = len(edges_work)
edges_work = edges_work[edges_work["highway_clean"].isin(WALK_WHITELIST)].copy()
print(f"After whitelist filter: {len(edges_work):,}  (removed {before - len(edges_work):,})")

# ---------------------------------------------------------------------------
# ACCESS FILTER
# Remove segments explicitly prohibited to pedestrians
# ---------------------------------------------------------------------------
def is_restricted(row):
    foot   = normalise_val(row.get("foot",   "")).lower()
    access = normalise_val(row.get("access", "")).lower()
    hw     = row["highway_clean"]
    if foot in ("no", "private"):
        return True
    if access in ("no", "private") and hw not in WALK_INFRA:
        return True
    return False

edges_work["_restricted"] = edges_work.apply(is_restricted, axis=1)
before = len(edges_work)
edges_work = edges_work[~edges_work["_restricted"]].drop(columns=["_restricted"]).copy()
print(f"After access filter:   {len(edges_work):,}  (removed {before - len(edges_work):,})")

# ---------------------------------------------------------------------------
# REBUILD GRAPH FROM FILTERED EDGES
# Preserves osmnx topology -- shared nodes and planarized intersections
# ---------------------------------------------------------------------------
print("\nRebuilding graph from filtered edges...")
node_ids_needed = set(edges_work["u"]).union(set(edges_work["v"]))
nodes_work = nodes[nodes.index.isin(node_ids_needed)].copy()

edges_indexed = edges_work.set_index(["u", "v", "key"])
G_filtered = ox.graph_from_gdfs(nodes_work, edges_indexed)

# ---------------------------------------------------------------------------
# EXTRACT LARGEST WEAKLY CONNECTED COMPONENT
# ---------------------------------------------------------------------------
n_components   = nx.number_weakly_connected_components(G_filtered)
largest        = max(nx.weakly_connected_components(G_filtered), key=len)
coverage       = len(largest) / len(nodes_work) * 100

print(f"  Weakly connected components: {n_components:,}")
print(f"  Nodes in largest component:  {len(largest):,} ({coverage:.1f}% of filtered nodes)")

if coverage < 80:
    print("  WARNING: Coverage below 80%. Check source data quality before continuing.")
elif coverage >= 95:
    print("  Coverage is excellent.")
else:
    print("  Coverage is acceptable.")

G_main = G_filtered.subgraph(largest).copy()
nodes_final, edges_final = ox.graph_to_gdfs(G_main)
edges_final = edges_final.reset_index()

print(f"\nFinal graph -- Nodes: {len(nodes_final):,}  Edges: {len(edges_final):,}")

# Highway type distribution.
# OSM uses 'highway' for ALL transport infrastructure, not just motor roads.
# Expect to see pedestrian types (steps, footway, cycleway, corridor, track)
# alongside road types (residential, primary, etc.) -- all are intentional.
print("\nHighway type distribution (OSM highway tag values):")
print(edges_final["highway"].apply(normalise_val).value_counts().to_string())


After whitelist filter: 1,382,261  (removed 1,331,469)
After access filter:   1,330,832  (removed 51,429)

Rebuilding graph from filtered edges...
  Weakly connected components: 1,834
  Nodes in largest component:  501,630 (98.1% of filtered nodes)
  Coverage is excellent.

Final graph — Nodes: 501,630  Edges: 1,314,375

Highway distribution:
highway
footway           460481
residential       325174
service           207687
tertiary           76161
cycleway           64350
path               40796
secondary          40767
unclassified       24954
track              24889
primary            18970
pedestrian         17874
steps               4950
living_street       2135
corridor            1922
primary_link        1300
secondary_link       899
tertiary_link        563
bridleway            429
elevator              74


## Step 6: Calculate WalkTime

**WalkTime** is calculated in minutes after reprojecting to **GDA2020 MGA
(zone and EPSG auto-selected in the CONFIG cell)** so that geometry length is in metres. Reprojecting before measuring
length is essential — WGS84 geometry lengths are in degrees and produce incorrect
walk times.

**Surface-adjusted speed** is applied only to dedicated walking infrastructure
(footways, paths, steps, cycleways, tracks). Applying surface penalties to road
centrelines would be incorrect — a pedestrian walking along a road shoulder walks
at normal pace regardless of the OSM surface tag on that road.

> **Note:** `normalise_val` and `WALK_INFRA` used in this cell are defined in Step 5.
> Cells must be run in order.


In [7]:
# Surface type -> effective walk speed (km/h)
# Applied only to dedicated walking infrastructure (WALK_INFRA defined in Step 5)
SURFACE_SPEED = {
    "asphalt": 5.0, "concrete": 5.0, "paved": 5.0,
    "paving_stones": 5.0, "compacted": 4.5,
    "fine_gravel": 4.5, "gravel": 4.0,
    "unpaved": 4.0, "dirt": 4.0, "grass": 4.0,
    "ground": 4.0, "mud": 3.0, "sand": 3.0
}

# ---------------------------------------------------------------------------
# REPROJECT TO GDA2020 MGA (zone determined in CONFIG cell)
# Geometry length must be in metres for accurate WalkTime calculation
# ---------------------------------------------------------------------------
print(f"Reprojecting to GDA2020 MGA Zone {_mga_zone} (EPSG:{EPSG})...")
edges_final = edges_final.to_crs(epsg=EPSG)

# ---------------------------------------------------------------------------
# SURFACE-ADJUSTED WALK SPEED AND WALKTIME
# ---------------------------------------------------------------------------
edges_final["highway_n"] = edges_final["highway"].apply(normalise_val)
edges_final["surface_n"] = edges_final["surface"].apply(normalise_val) \
    if "surface" in edges_final.columns else ""

def get_speed(row):
    # Surface penalty only on dedicated walking infrastructure
    if row["highway_n"] in WALK_INFRA:
        return SURFACE_SPEED.get(
            str(row.get("surface_n", "")).lower().strip(),
            DEFAULT_WALK_KMH
        )
    return DEFAULT_WALK_KMH

edges_final["walk_speed"]     = edges_final.apply(get_speed, axis=1)
edges_final["shape_length_m"] = edges_final.geometry.length
edges_final["WalkTime"]       = (
    edges_final["shape_length_m"] / 1000
) / edges_final["walk_speed"] * 60

# ---------------------------------------------------------------------------
# LENGTH TRIM
# Remove geometry slivers (< 0.5 m) and very long rural tracks
# ---------------------------------------------------------------------------
before = len(edges_final)
edges_final = edges_final[
    (edges_final["shape_length_m"] >= 0.5) &
    (edges_final["shape_length_m"] <= MAX_SEGMENT_M)
].copy()
print(f"After length trim (0.5 m – {MAX_SEGMENT_M} m): {len(edges_final):,}  "
      f"(removed {before - len(edges_final):,})")

print(f"\nWalkTime stats (minutes):")
print(f"  Min:    {edges_final['WalkTime'].min():.3f}")
print(f"  Median: {edges_final['WalkTime'].median():.2f}")
print(f"  Max:    {edges_final['WalkTime'].max():.2f}")


Reprojecting to GDA2020 MGA Zone 56 (EPSG:7856)...
After length trim (0.5 m – 5000 m): 1,313,746  (removed 629)

WalkTime stats (minutes):
  Min:    0.006
  Median: 0.35
  Max:    72.92


## Step 7: Export to GeoPackage and File Geodatabase

The pipeline writes to GeoPackage first, then imports to the File Geodatabase
via arcpy. The GeoPackage is your safety net — if anything goes wrong in the
GDB import, you can reimport without rerunning Steps 4–6.

A dedicated **Feature Dataset** (`OSM_Transportation`) is created inside the GDB.
ArcGIS Network Datasets must live inside a Feature Dataset — they cannot be created
in the GDB root.

The `arcpy.env.workspace = TEMP_GPKG` pattern is required because arcpy does not
support the `file.gpkg|layername=layer` syntax for GeoPackage layer references.
Setting the workspace to the GPKG file allows the layer to be referenced by name.

**Why import to a temp name then rename?**

ArcGIS File GDB raises **Error 002851** if a feature class name already exists
anywhere in the geodatabase — even if you deleted it a line earlier. `Delete()`
returns success but the GDB catalog holds the name registration until the next
full catalog refresh. Importing to `OSM_Walk_Segments_tmp` first avoids this:
the temp name is guaranteed free, the import succeeds, then the old copy is
deleted cleanly and the temp is renamed.


In [8]:
import os
import arcpy

FDS_PATH      = os.path.join(GDB, FDS_NAME)
FC_EDGES_PATH = os.path.join(FDS_PATH, FC_EDGES_NAME)
FC_ROOT_PATH  = os.path.join(GDB, FC_EDGES_NAME)

# ---------------------------------------------------------------------------
# CREATE GDB AND FEATURE DATASET IF NEEDED
# ---------------------------------------------------------------------------
if not arcpy.Exists(GDB):
    arcpy.management.CreateFileGDB(os.path.dirname(GDB), os.path.basename(GDB))
    print(f'Created GDB: {GDB}')
else:
    print(f'GDB exists: {GDB}')

if not arcpy.Exists(FDS_PATH):
    arcpy.management.CreateFeatureDataset(GDB, FDS_NAME, arcpy.SpatialReference(EPSG))
    print(f'Created Feature Dataset: {FDS_NAME}  (EPSG:{EPSG})')
else:
    print(f'Feature Dataset exists: {FDS_NAME}')

# ---------------------------------------------------------------------------
# WRITE GEOPACKAGE (intermediate safety net)
# ---------------------------------------------------------------------------
cols = [
    'osmid', 'name', 'highway_n', 'surface_n',
    'walk_speed', 'WalkTime', 'shape_length_m', 'geometry'
]
cols = [c for c in cols if c in edges_final.columns]
edges_export = edges_final[cols].copy().rename(columns={
    'highway_n': 'highway',
    'surface_n': 'surface'
})
print()
print(f'Writing GeoPackage: {TEMP_GPKG}')
edges_export.to_file(TEMP_GPKG, driver='GPKG', layer='walk_segments')
print(f'  {len(edges_export):,} features written')

# ---------------------------------------------------------------------------
# IMPORT TO GDB — temp-name pattern to avoid Error 002851
#
# Error 002851 occurs when the GDB catalog still holds a name registration
# after Delete() returns — a File GDB race condition. Fix: import to a temp
# name first (no conflict possible), then delete the old copy, then rename.
# ---------------------------------------------------------------------------
TEMP_FC_NAME = FC_EDGES_NAME + '_tmp'
TEMP_FC_PATH = os.path.join(FDS_PATH, TEMP_FC_NAME)

# Clear any leftover temp from a previous failed run
if arcpy.Exists(TEMP_FC_PATH):
    arcpy.management.Delete(TEMP_FC_PATH)
    print(f'Cleared leftover temp: {TEMP_FC_NAME}')

# Step 1: import to temp name — guaranteed free, no naming conflict possible
print()
print(f'Importing to {TEMP_FC_NAME}...')
arcpy.env.workspace = TEMP_GPKG
arcpy.conversion.FeatureClassToFeatureClass(
    in_features = 'walk_segments',
    out_path    = FDS_PATH,
    out_name    = TEMP_FC_NAME
)
arcpy.env.workspace = None

# Step 2: delete all existing instances of the final name
# (safe now — we are not overwriting the target, just removing old copies)
for old_path in [FC_EDGES_PATH, FC_ROOT_PATH]:
    if arcpy.Exists(old_path):
        arcpy.management.Delete(old_path)
        print(f'Deleted existing: {old_path}')

# Step 3: copy temp to final name using explicit full paths, then delete temp.
# arcpy.management.Rename resolves output against arcpy.env.workspace, which
# may point to a different GDB from a previous cell run. Copy with full paths
# bypasses workspace resolution entirely.
arcpy.env.workspace = None
FC_EDGES_PATH = os.path.join(FDS_PATH, FC_EDGES_NAME)
arcpy.management.Copy(TEMP_FC_PATH, FC_EDGES_PATH)
arcpy.management.Delete(TEMP_FC_PATH)

count = int(arcpy.management.GetCount(FC_EDGES_PATH)[0])
print()
print(f'Imported: {count:,} features')
print(f'Location: {FC_EDGES_PATH}')


Created GDB: C:\Work\Brisbane_GTFS\SEQ_Network\Sydney_Walk_Network.gdb
Created Feature Dataset: OSM_Transportation  (EPSG:7856)

Writing GeoPackage: C:\Work\Brisbane_GTFS\SEQ_Network\sydney_walk.gpkg
  1,313,746 features written

Importing to OSM_Walk_Segments_tmp...

Imported: 1,313,746 features
Location: C:\Work\Brisbane_GTFS\SEQ_Network\Sydney_Walk_Network.gdb\OSM_Transportation\OSM_Walk_Segments


## Step 8: Create and Build the Network Dataset

The Network Dataset is created programmatically from the template XML included
with this notebook (`osm_walk_nd_template.xml`), then built. No manual steps in
the ArcGIS Pro UI are required.

**What the template encodes:**

| Setting | Value |
|---|---|
| Edge source | `OSM_Walk_Segments` |
| Elevation model | None — not needed for osmnx-derived topology |
| **Length** cost | Geometry length in metres (Field: `Shape`) |
| **WalkTime** cost | Minutes (Field evaluator: `!WalkTime!`) |
| **Walking** travel mode | WalkTime impedance, all U-turns, no hierarchy |

> **Template location:** `osm_walk_nd_template.xml` must be in the same folder as
> this notebook. The path is resolved automatically via `os.getcwd()` in the CONFIG cell.
>
> **Source name:** `FC_EDGES_NAME` must remain `OSM_Walk_Segments` to match the template.
> If you rename the feature class, export a new template from ArcGIS Pro first:
> Catalog pane → right-click the Network Dataset → **Export Network Dataset Template**.


In [9]:
import arcpy
import os

FDS_PATH      = os.path.join(GDB, FDS_NAME)
ND_PATH       = os.path.join(GDB, FDS_NAME, ND_NAME)
FC_EDGES_PATH = os.path.join(FDS_PATH, FC_EDGES_NAME)

# ---------------------------------------------------------------------------
# VALIDATE TEMPLATE
# ---------------------------------------------------------------------------
if not os.path.exists(TEMPLATE_XML):
    raise FileNotFoundError(
        f"Network Dataset template not found: {TEMPLATE_XML}\n"
        "Ensure osm_walk_nd_template.xml is in the same folder as this notebook."
    )

# ---------------------------------------------------------------------------
# CREATE NETWORK DATASET FROM TEMPLATE
# Template encodes costs (Length, WalkTime) and Walking travel mode.
# Spatial reference is taken from the Feature Dataset, not the template.
# ---------------------------------------------------------------------------
if arcpy.Exists(ND_PATH):
    arcpy.management.Delete(ND_PATH)
    print(f"Deleted existing Network Dataset: {ND_NAME}")

print(f"Creating Network Dataset from template...")
# CreateNetworkDatasetFromTemplate is the correct tool when using a template XML.
# CreateNetworkDataset does not expose a template_file parameter in ArcGIS Pro 3.x.
arcpy.na.CreateNetworkDatasetFromTemplate(
    TEMPLATE_XML,  # template_file — defines costs, travel mode, source name
    FDS_PATH       # output_feature_dataset
)
print(f"  Created: {ND_PATH}")

# ---------------------------------------------------------------------------
# BUILD NETWORK DATASET
# ---------------------------------------------------------------------------
print(f"\nBuilding network — expect 1–3 minutes...")
arcpy.na.BuildNetwork(ND_PATH)
print("Build complete.")

# ---------------------------------------------------------------------------
# VERIFY
# ---------------------------------------------------------------------------
desc  = arcpy.Describe(ND_PATH)
count = int(arcpy.management.GetCount(FC_EDGES_PATH)[0])
print(f"\nNetwork Dataset: {desc.name}")
print(f"  Edge sources:     {[s.name for s in desc.edgeSources]}")
print(f"  Junction sources: {[s.name for s in desc.junctionSources]}")
print(f"  Edge count:       {count:,}")
print(f"\nReady for Service Area and Route analysis.")
print(f"Quick test: Analysis tab → Service Area | Travel Mode: Walking | Cutoffs: 5 10 15")

Creating Network Dataset from template...
  Created: C:\Work\Brisbane_GTFS\SEQ_Network\Sydney_Walk_Network.gdb\OSM_Transportation\OSM_Walk_ND

Building network — expect 1–3 minutes...
Build complete.

Network Dataset: OSM_Walk_ND
  Edge sources:     ['OSM_Walk_Segments']
  Junction sources: ['OSM_Walk_ND_Junctions']
  Edge count:       1,313,746

Ready for Service Area and Route analysis.
Quick test: Analysis tab → Service Area | Travel Mode: Walking | Cutoffs: 5 10 15


## Known Limitations

### OSM Coverage Gaps
OSM volunteer coverage is uneven. Newer suburban developments often have sparse
footpath data. Walk catchments in these areas will be smaller than reality.
This is a source data limitation, not a processing issue. Coverage improves as
volunteers map new areas — re-downloading the PBF and rerunning the notebook
periodically will pick up improvements.

### WalkTime Models Physical Accessibility, Not Route Preference
The 5 km/h flat-speed model answers *“can a pedestrian physically reach this point
within X minutes?”* — the correct question for catchment analysis. It does not
model route preference (pedestrians preferring footways over road shoulders).
For navigation or pedestrian comfort modelling, a resistance-based approach would
be needed.

### Network Built from a Point-in-Time OSM Snapshot
The Australia PBF file is a snapshot. New footpaths, crossings, and road changes
made in OSM after the download date are not reflected. Re-download the PBF and
rerun this notebook periodically to keep the network current.

### osmconvert Clips by Bounding Box, Not Administrative Boundary
Features near the bbox edges may extend slightly beyond your intended study area.
For analysis requiring a precise boundary, clip the final feature class to an
administrative polygon after import.

---

## Credits and Data Sources

| Source | Licence | URL |
|---|---|---|
| OpenStreetMap contributors | ODbL | https://www.openstreetmap.org |
| Geofabrik Australia extract | ODbL | https://download.geofabrik.de/australia-oceania/australia.html |
| osmnx | MIT | https://osmnx.readthedocs.io |
| osmconvert | GPLv3 | https://wiki.openstreetmap.org/wiki/Osmconvert |

> © OpenStreetMap contributors. Data is available under the Open Database Licence:
> https://opendatacommons.org/licenses/odbl/
>
> When publishing analysis based on this network, acknowledge OpenStreetMap contributors
> and Geofabrik as the data source.


## Step 9: Clean Up Intermediate Files

The pipeline creates two intermediate files that are no longer needed once
the Network Dataset is built and verified:

| File | Purpose | Safe to delete? |
|---|---|---|
| `{city}_walk.osm` | osmconvert-clipped OSM extract | Yes — regenerates in 3–5 min from the PBF |
| `{city}_walk.gpkg` | GeoPackage staging file | Yes — authoritative data is now in the GDB |

The original Australia PBF is **not deleted** — it is reused when building
networks for other cities and takes ~900 MB to redownload.

Run this cell after verifying the Network Dataset produces correct results.


In [ ]:
import os

to_delete = [CLIPPED_OSM, TEMP_GPKG]

for path in to_delete:
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted:   {path}")
    else:
        print(f"Not found: {path}")

print(f"\nKept: {AUSTRALIA_PBF}")
print("Cleanup complete.")
